# Lecture 06 — NumPy

*Doing maths on whole columns of numbers at once: concentrations, descriptors and a UV–Vis spectrum.*

## Learning objectives

By the end of this lecture you will be able to:

- explain why **NumPy arrays** beat plain lists for numerical work;
- create arrays with `np.array`, `np.arange`, `np.linspace`, `np.zeros` / `np.ones`;
- do **element-wise** arithmetic and understand **broadcasting**;
- **index**, **slice** and apply **boolean masks** to arrays;
- compute **aggregations**: `mean`, `std`, `min`, `max`, `sum`, `argmax`.

## Recap of Lecture 05

- Files are opened safely with **`with`**; CSVs read cleanly via the `csv` module or `pandas`.
- We screened `data/molecules.csv` with the rule of five and wrote out a subset.
- Numbers read from text files arrive as **strings** and need converting.

## Why NumPy?

A Python **list** can hold numbers, but doing maths on it is clumsy — to double every value you need a loop. **NumPy** gives you the **array**: a list-like container built for numbers, where maths applies to *every element at once* and runs fast. Watch the difference:

In [ ]:
import numpy as np

# A plain list: multiplying by 2 does NOT do what a chemist wants...
concs_list = [0.1, 0.2, 0.3]
print("list * 2:", concs_list * 2)        # it repeats the list!

# A NumPy array: maths applies element by element.
concs = np.array([0.1, 0.2, 0.3])
print("array * 2:", concs * 2)            # every value doubled

That element-wise behaviour is the whole point. No loop, no fuss — and for big datasets it is dramatically faster.

## Creating arrays

A few standard ways to make arrays:

- `np.array([...])` — from a list you already have.
- `np.arange(start, stop, step)` — evenly spaced, like `range` (stops *before* `stop`).
- `np.linspace(start, stop, n)` — `n` points evenly spaced, **including** both ends.
- `np.zeros(n)` / `np.ones(n)` — arrays filled with 0.0 or 1.0.

In [ ]:
print(np.arange(0, 1, 0.25))         # 0, 0.25, 0.5, 0.75
print(np.linspace(0, 1, 5))          # 5 points from 0 to 1 inclusive
print(np.zeros(3))
print(np.ones(3))

## Element-wise arithmetic and broadcasting

> **🧪 Chemistry aside — the Beer–Lambert law**
>
> The **Beer–Lambert law** relates the absorbance of a solution to its concentration: `A = ε · c · l`, where `ε` is the molar absorptivity (how strongly the substance absorbs), `c` is the concentration, and `l` is the path length of the cuvette. Given a list of concentrations, we can get all their absorbances in one line.

Here a single number (`epsilon * path_length`) multiplies a whole array — NumPy "broadcasts" the scalar across every element.

In [ ]:
concentrations = np.linspace(0.0, 1.0e-4, 6)   # mol/L (0 to 100 micromolar)
epsilon = 12000      # molar absorptivity, L/(mol*cm)
path_length = 1.0    # cm

absorbances = epsilon * concentrations * path_length   # Beer-Lambert, all at once
print("Concentrations:", concentrations)
print("Absorbances:   ", absorbances)

### 🔬 Try it yourself

Make an array of concentrations from `0` to `5e-5 mol/L` (that is 50 µM) with **11** points (`np.linspace`). Using `ε = 8000` and a `1 cm` path length, compute the absorbances in one line and print them.

In [ ]:
# Your code here.

**Solution**

In [ ]:
c = np.linspace(0, 5e-5, 11)
a = 8000 * c * 1.0
print(a)

## Indexing, slicing and boolean masks

Arrays index and slice just like lists (still zero-based). The new superpower is the **boolean mask**: compare an array to a value and you get an array of `True`/`False`, which you can use to *select* elements.

In [ ]:
print(absorbances[0])        # first element
print(absorbances[1:4])      # a slice

# A boolean mask: which absorbances exceed 0.5?
mask = absorbances > 0.5
print(mask)                  # array of True/False
print(absorbances[mask])     # keep only the elements where the mask is True

## Aggregations: summarising an array

NumPy arrays have handy methods to summarise them: `.mean()`, `.std()` (standard deviation), `.min()`, `.max()`, `.sum()`. Let us use the molecule dataset. We will load it with pandas (from Lecture 05) and pull the molar-mass column out as a NumPy array.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/molecules.csv")
masses = df["molar_mass"].to_numpy()      # a column -> a NumPy array
print("Molar masses:", masses)
print(f"mean  = {masses.mean():.2f} g/mol")
print(f"std   = {masses.std():.2f} g/mol")
print(f"min   = {masses.min():.2f} g/mol")
print(f"max   = {masses.max():.2f} g/mol")

### 🔬 Try it yourself

Pull the `logp` column out as a NumPy array. (1) Print its **mean** and **standard deviation**. (2) Use a **boolean mask** to print only the logP values **above 1.0**.

In [ ]:
# Your code here.

**Solution**

In [ ]:
logp = df["logp"].to_numpy()
print(f"mean logP = {logp.mean():.2f}")
print(f"std  logP = {logp.std():.2f}")
print("logP above 1.0:", logp[logp > 1.0])

## Processing a spectrum

Now a classic array job: a **UV–Vis spectrum**. The file `data/uvvis_spectrum.csv` has two columns, `wavelength_nm` and `absorbance`. We load them into arrays and do real analysis.

In [ ]:
spectrum = pd.read_csv("../data/uvvis_spectrum.csv")
wavelength = spectrum["wavelength_nm"].to_numpy()
absorbance = spectrum["absorbance"].to_numpy()
print("Number of points:", len(wavelength))
print("Wavelength range:", wavelength.min(), "to", wavelength.max(), "nm")

**Where is the peak?** `argmax` gives the *index* of the largest value; use it to look up the wavelength there.

In [ ]:
peak_index = absorbance.argmax()
print(f"Peak absorbance {absorbance[peak_index]:.3f} at {wavelength[peak_index]:.0f} nm")

**Normalising** is just element-wise division — rescale so the maximum becomes 1:

In [ ]:
normalised = absorbance / absorbance.max()
print("New maximum:", normalised.max())
print("First few normalised values:", normalised[:5])

**Slicing a window** with a boolean mask — keep only the 250–290 nm region:

In [ ]:
window = (wavelength >= 250) & (wavelength <= 290)
print("Points in the 250-290 nm window:", window.sum())
print("Mean absorbance there:", absorbance[window].mean().round(3))

## ⚗️ With RDKit — build a descriptor array, then do statistics

Here RDKit (which makes molecule **objects**) and NumPy (which crunches **numbers**) team up. We loop the SMILES through RDKit to compute three descriptors per molecule — molar mass, logP and **TPSA** — and stack them into a NumPy array to analyse.

> **🧪 Chemistry aside — TPSA.** The **topological polar surface area** (TPSA) estimates the surface area of a molecule taken up by polar (nitrogen- and oxygen-containing) parts. It correlates with how easily a drug crosses membranes — another quick descriptor RDKit gives for free.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

rows = []
for smiles in df["smiles"]:
    mol = Chem.MolFromSmiles(smiles)
    rows.append([Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol)])

descriptors = np.array(rows)        # a 2D array: one row per molecule, three columns
print("Shape (rows, columns):", descriptors.shape)
print(descriptors.round(2))

With a 2D array, `axis=0` means "down each column". So we can summarise all three descriptors at once:

In [ ]:
column_names = ["MolWt", "MolLogP", "TPSA"]
means = descriptors.mean(axis=0)        # mean of each column
for name, value in zip(column_names, means):
    print(f"mean {name:<8} = {value:.2f}")

### 🔬 Try it yourself

Using the `descriptors` array: (1) find the **maximum TPSA** (the third column, index `2`); (2) find which molecule it belongs to. Hint: `descriptors[:, 2]` is the whole TPSA column, and `argmax` gives its index; look that index up in `df["name"]`.

In [ ]:
# Your code here.

**Solution**

In [ ]:
tpsa = descriptors[:, 2]
i = tpsa.argmax()
print(f"Highest TPSA is {tpsa[i]:.2f} for {df['name'].iloc[i]}")

## Key takeaways

- A **NumPy array** does maths **element-wise** and fast — no loops needed.
- Create arrays with `np.array`, `np.arange`, `np.linspace`, `np.zeros`/`np.ones`.
- A scalar applied to an array **broadcasts** across every element (e.g. Beer–Lambert in one line).
- **Boolean masks** (`arr[arr > x]`) select elements that pass a test.
- Summarise with `mean`, `std`, `min`, `max`, `sum`; find the peak position with `argmax`.

## Looking ahead

Next lecture — **Plotting** — we turn all these numbers into figures: the UV–Vis spectrum, a calibration curve, and a property scatter — with properly labelled, unit-bearing axes.